<a href="https://colab.research.google.com/github/ML-HW-SYS/a2-AaronGoldblatt/blob/main/4_model_conversion_structured_pruned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **4. Convert PyTorch Model to TFLite**

## 4.0 Setup GDrive and Git

### Import GitHub dependencies and mount Google Drive
You should now see your GitHub files under the folder icon to the left. Make sure to include your own token.

In [2]:
# Mount google drive
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [3]:
# Make sure your token is stored in a txt file at the location below.
# This way there is no risk that you will push it to your repo
# Never share your token with anyone, it is basically your github password!
with open('/content/gdrive/MyDrive/ece5545/token.txt') as f:
    token = f.readline().strip()
# Use another file to store your github username
with open('/content/gdrive/MyDrive/ece5545/git_username.txt') as f:
    handle = f.readline().strip()

In [4]:
# Clone your github repo
YOUR_TOKEN = token
YOUR_HANDLE = handle
BRANCH = "main"

%mkdir /content/gdrive/MyDrive/ece5545
%cd /content/gdrive/MyDrive/ece5545
!git clone https://{YOUR_TOKEN}@github.com/ML-HW-SYS/a2-{YOUR_HANDLE}.git
%cd /content/gdrive/MyDrive/ece5545/a2-{YOUR_HANDLE}
!git checkout {BRANCH}
!git pull
%cd /content/gdrive/MyDrive/ece5545

PROJECT_ROOT = f"/content/gdrive/MyDrive/ece5545/a2-{YOUR_HANDLE}"

mkdir: cannot create directory ‘/content/gdrive/MyDrive/ece5545’: File exists
/content/gdrive/MyDrive/ece5545
fatal: destination path 'a2-AaronGoldblatt' already exists and is not an empty directory.
/content/gdrive/MyDrive/ece5545/a2-AaronGoldblatt
Already on 'main'
Your branch is up to date with 'origin/main'.
Already up to date.
/content/gdrive/MyDrive/ece5545


In [4]:
# This extension reloads all imports before running each cell
%load_ext autoreload
%autoreload 2

### Install required packages

In [5]:
# Install ONNX
!pip3 install onnx

# Install tensorflow-addons
!pip3 install tensorflow-addons
!pip3 install tensorflow-probability

!pip3 install torchaudio

# Install onnx-tensorflow
# !git clone https://github.com/onnx/onnx-tensorflow.git
# !cd onnx-tensorflow && pip3 install -e .
!pip3 install onnx-tf==1.9.0
!pip install 'keras<3.0.0'
!pip install tensorflow==2.12.0

# [Alert] If there is a popup that tells you restart session.
# No worries! Just click restart on your session, rerun the code blocks above, and you're all set.
# No need to rerun this installment block at all.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 kB 21.0 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.4.2
    Uninstalling typeguard-4.4.2:
      Successfully uninstalled typeguard-4.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inflect 7.5.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━

### Import code dependencies

In [5]:
# Import libraries
import sys,os

# Add path to onnx-tensorflow
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'onnx-tensorflow'))
sys.path.insert(0, PROJECT_ROOT)

import onnx
import torch
import torch.nn as nn
import numpy as np
import os
import tensorflow as tf
print(f"TensorFlow Version is: {tf.__version__}")
from onnx_tf.backend import prepare

# Import constants to use constants defined for training
from src.constants import *
import src.data_proc as data_proc

TensorFlow Version is: 2.12.0


/usr/local/lib/python3.11/dist-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.13.0 and strictly below 2.16.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.12.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're u

Model folders are created, 
PyTorch models will be saved in /content/gdrive/MyDrive/ece5545/models/torch_models, 
ONNX models will be saved in /content/gdrive/MyDrive/ece5545/models/onnx_models, 
TensorFlow Saved Models will be saved in /content/gdrive/MyDrive/ece5545/models/tf_models, 
TensorFlow Lite models will be saved in /content/gdrive/MyDrive/ece5545/models/tflite_models, 
TensorFlow Lite Micro models will be saved in /content/gdrive/MyDrive/ece5545/models/micro_models.


## 4.1 Define the Model


**Replace the torch_path model with the model you created in the last section.**

You can find the name of your file in `TORCH_DIR` under the folder icon to the left.

In [6]:
!ls {TORCH_DIR}

 finetuned_structured_pruning_model_threshold_0.25.pt   fp32_checkpoint.pt.old
 finetuned_structured_pruning_model_threshold_0.45.pt   fp32_finetune_checkpoint.pt
 finetuned_structured_pruning_model_threshold_0.6.pt    fp32_finetune_checkpoint.pt.old
 finetuned_structured_pruning_model_threshold_0.75.pt  '(QAT4bit)quant_0.pt'
 finetuned_structured_pruning_model_threshold_0.85.pt   quant_checkpoint.pt
 finetuned_structured_pruning_model_threshold_0.95.pt   quant_checkpoint.pt.old
 fp32_checkpoint.pt				        tinyconv_float32_init_seed0_92.25%_0.pt


Please use a checkpoint from the cell above and place it in the cell below for the `torch_path`

In [38]:
from src.networks import TinyConv
import os.path as osp

# Create audio processor
audio_processor = data_proc.AudioProcessor(data_dir=DATASET_DIR) # DATASET_DIR is defined in constants

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create a full precision (float32) TinyConv model
model_fp32 = TinyConv(model_settings=audio_processor.model_settings, \
    n_input=1, n_output=audio_processor.num_labels)
model_fp32.to(device)

# TODO: Replace me!
# torch_path = osp.join(TORCH_DIR, "finetuned_structured_pruning_model_threshold_0.25.pt")
# torch_path = osp.join(TORCH_DIR, "finetuned_structured_pruning_model_threshold_0.45.pt")
# torch_path = osp.join(TORCH_DIR, "finetuned_structured_pruning_model_threshold_0.6.pt")
# torch_path = osp.join(TORCH_DIR, "finetuned_structured_pruning_model_threshold_0.75.pt")
# torch_path = osp.join(TORCH_DIR, "finetuned_structured_pruning_model_threshold_0.85.pt")
torch_path = osp.join(TORCH_DIR, "finetuned_structured_pruning_model_threshold_0.95.pt")

# Load model
model_fp32.load_state_dict(torch.load(torch_path))
model_fp32

<ipython-input-38-f9fb03161348>:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_fp32.load_state_dict(torch.load(torch_path))


TinyConv(
  (conv_reshape): Reshape(output_shape=(-1, 1, 49, 40))
  (conv): Conv2d(1, 8, kernel_size=(10, 8), stride=(2, 2), padding=(5, 3))
  (relu): ReLU()
  (dropout): Dropout(p=0.5, inplace=False)
  (fc_reshape): Reshape(output_shape=(-1, 4000))
  (fc): Linear(in_features=4000, out_features=4, bias=True)
  (softmax): Softmax(dim=1)
)

## 4.2 Convert from PyTorch to TFLite
This is the last step before deploying your model to your TinyML device.

In [39]:
from src.train_val_test_utils import choose_name
# Choose a path
# You can also define your own path
# file_name = choose_name("micro_structured_pruned_threshold_0_25")
# file_name = choose_name("micro_structured_pruned_threshold_0_45")
# file_name = choose_name("micro_structured_pruned_threshold_0_6")
# file_name = choose_name("micro_structured_pruned_threshold_0_75")
# file_name = choose_name("micro_structured_pruned_threshold_0_85")
file_name = choose_name("micro_structured_pruned_threshold_0_95")

The model's will be stored as: 
 micro_structured_pruned_threshold_0_95_0


### Convert to ONNX

In [40]:
# Create a new TinyConv model
ONNX_PATH = os.path.join(ONNX_DIR, f'{file_name}.onnx')
TF_PATH = os.path.join(TF_DIR, f'{file_name}')

# Create a random input for model export
dummy_input = torch.autograd.Variable(torch.randn(1, 1960))

# Export model as .onnx file
torch.onnx.export(model_fp32.cpu(), dummy_input, ONNX_PATH, input_names=['input'], output_names=['output'])

# Load onnx model into a tensorflow model
onnx_model = onnx.load(ONNX_PATH)
tf_rep = prepare(onnx_model)

# Export model as .pb file

tf_rep.export_graph(TF_PATH)

### Convert to TFLite

In [41]:
FLOAT_MODEL_TFLITE = os.path.join(TFLITE_DIR, f'float_{file_name}.tflite')
MODEL_TFLITE = os.path.join(TFLITE_DIR, f'quant_{file_name}.tflite')

float_converter = tf.lite.TFLiteConverter.from_saved_model(TF_PATH)
float_tflite_model = float_converter.convert()
float_tflite_model_size = open(FLOAT_MODEL_TFLITE, "wb").write(float_tflite_model)
print("Float model is %d bytes" % float_tflite_model_size)

converter = tf.lite.TFLiteConverter.from_saved_model(TF_PATH)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
def representative_dataset_gen():
    for i in range(100):
        data, _ = audio_processor.get_data(
            1, i*1, BACKGROUND_FREQUENCY, BACKGROUND_VOLUME_RANGE, TIME_SHIFT_MS, 'testing')
        flattened_data = np.array(data.flatten(), dtype=np.float32).reshape([1, 1960])
        yield [flattened_data]
converter.representative_dataset = representative_dataset_gen

tflite_model = converter.convert()
tflite_model_size = open(MODEL_TFLITE, "wb").write(tflite_model)
print("Quantized model is %d bytes" % tflite_model_size)

Float model is 69296 bytes
Quantized model is 20008 bytes


Notice how the quantized model is less than a third of the size of the float model? We'll next see how this reduction in size affects model accuracy.

### TFLite Model Accuracy

In [ ]:
# # Helper function to run inference
# def run_tflite_inference_test_set(tflite_model_path, model_type="Float"):
#     # Load test data
#     test_data, test_labels = audio_processor.get_data(
#           -1, 0, BACKGROUND_FREQUENCY, BACKGROUND_VOLUME_RANGE,
#           TIME_SHIFT_MS, 'testing')
#     test_data = np.expand_dims(test_data, axis=1).astype(np.float32)

#     # Initialize the interpreter
#     interpreter = tf.lite.Interpreter(tflite_model_path)
#     interpreter.allocate_tensors()
#     input_details = interpreter.get_input_details()[0]
#     output_details = interpreter.get_output_details()[0]

#     # For quantized models, manually quantize the input data from float to integer
#     if model_type == "Quantized":
#         input_scale, input_zero_point = input_details["quantization"]
#         test_data = test_data / input_scale + input_zero_point
#         test_data = test_data.astype(input_details["dtype"])

#     # Evaluate the predictions
#     correct_predictions = 0
#     for i in range(len(test_data)):
#         interpreter.set_tensor(input_details["index"], test_data[i])
#         interpreter.invoke()
#         output = interpreter.get_tensor(output_details["index"])[0]
#         top_prediction = output.argmax()
#         correct_predictions += (top_prediction == test_labels[i])

#     print('%s model accuracy is %f%% (Number of test samples=%d)' % (
#         model_type, (correct_predictions * 100) / len(test_data), len(test_data)))

In [ ]:
# # Compute float model accuracy
# run_tflite_inference_test_set(FLOAT_MODEL_TFLITE)

# # Compute quantized model accuracy
# run_tflite_inference_test_set(MODEL_TFLITE, model_type='Quantized')

Float model accuracy is 92.470757% (Number of test samples=1368)
Quantized model accuracy is 91.739769% (Number of test samples=1368)


As you can see, quantizing the model hardly affects the model accuracy in this case and makes it much more easily deployable to a tiny machine.

We will now create the byte version of our model needed for deployment.

**Replace the name of file in MODEL_TFLITE with the *quant* model you saved earlier in this notebook.**

You can find the name of your file in "{PROJECT_ROOT}/models/tflite_models" under the folder icon to the left.

### TFLite Micro Conversion

In [42]:
!apt-get update && apt-get -qq install xxd

# Replace me!
# MODEL_TFLITE = TFLITE_DIR + '/quant_micro_structured_pruned_threshold_0_25_0.tflite'
# MODEL_TFLITE_MICRO = MICRO_DIR+'/micro_structured_pruned_threshold_0_25.cc'

# MODEL_TFLITE = TFLITE_DIR + '/quant_micro_structured_pruned_threshold_0_45_0.tflite'
# MODEL_TFLITE_MICRO = MICRO_DIR+'/micro_structured_pruned_threshold_0_45.cc'

# MODEL_TFLITE = TFLITE_DIR + '/quant_micro_structured_pruned_threshold_0_6_0.tflite'
# MODEL_TFLITE_MICRO = MICRO_DIR+'/micro_structured_pruned_threshold_0_6.cc'

# MODEL_TFLITE = TFLITE_DIR + '/quant_micro_structured_pruned_threshold_0_75_0.tflite'
# MODEL_TFLITE_MICRO = MICRO_DIR+'/micro_structured_pruned_threshold_0_75.cc'

# MODEL_TFLITE = TFLITE_DIR + '/quant_micro_structured_pruned_threshold_0_85_0.tflite'
# MODEL_TFLITE_MICRO = MICRO_DIR+'/micro_structured_pruned_threshold_0_85.cc'

MODEL_TFLITE = TFLITE_DIR + '/quant_micro_structured_pruned_threshold_0_95_0.tflite'
MODEL_TFLITE_MICRO = MICRO_DIR+'/micro_structured_pruned_threshold_0_95.cc'

!xxd -i {MODEL_TFLITE} > {MODEL_TFLITE_MICRO}
REPLACE_TEXT = MODEL_TFLITE.replace('/', '_').replace('.', '_')

!sed -i 's/'{REPLACE_TEXT}'/g_model/g' {MODEL_TFLITE_MICRO}

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:3 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:4 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


You can find the below bites in the "{PROJECT_ROOT}/models/micro_models" folder. If you do not see an output after the next cell, make sure you replaced the file name  of {MODEL_TFLITE} in the cell above with your quant model created earlier in this notebook.

In [12]:
!cat {MICRO_DIR}/micro_structured_pruned_threshold_0_25.cc

unsigned char g_model[] = {
  0x1c, 0x00, 0x00, 0x00, 0x54, 0x46, 0x4c, 0x33, 0x14, 0x00, 0x20, 0x00,
  0x1c, 0x00, 0x18, 0x00, 0x14, 0x00, 0x10, 0x00, 0x0c, 0x00, 0x00, 0x00,
  0x08, 0x00, 0x04, 0x00, 0x14, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00,
  0x80, 0x00, 0x00, 0x00, 0xd8, 0x00, 0x00, 0x00, 0xc8, 0x43, 0x00, 0x00,
  0xd8, 0x43, 0x00, 0x00, 0x64, 0x4d, 0x00, 0x00, 0x03, 0x00, 0x00, 0x00,
  0x01, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00, 0x4e, 0xbb, 0xff, 0xff,
  0x0c, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00, 0x38, 0x00, 0x00, 0x00,
  0x0f, 0x00, 0x00, 0x00, 0x73, 0x65, 0x72, 0x76, 0x69, 0x6e, 0x67, 0x5f,
  0x64, 0x65, 0x66, 0x61, 0x75, 0x6c, 0x74, 0x00, 0x01, 0x00, 0x00, 0x00,
  0x04, 0x00, 0x00, 0x00, 0x9c, 0xff, 0xff, 0xff, 0x11, 0x00, 0x00, 0x00,
  0x04, 0x00, 0x00, 0x00, 0x06, 0x00, 0x00, 0x00, 0x6f, 0x75, 0x74, 0x70,
  0x75, 0x74, 0x00, 0x00, 0x01, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00,
  0x0e, 0xbc, 0xff, 0xff, 0x04, 0x00, 0x00, 0x00, 0x05, 0x00, 0x00, 0x00,
  0x69, 0x

In [19]:
!cat {MICRO_DIR}/micro_structured_pruned_threshold_0_45.cc

unsigned char g_model[] = {
  0x1c, 0x00, 0x00, 0x00, 0x54, 0x46, 0x4c, 0x33, 0x14, 0x00, 0x20, 0x00,
  0x1c, 0x00, 0x18, 0x00, 0x14, 0x00, 0x10, 0x00, 0x0c, 0x00, 0x00, 0x00,
  0x08, 0x00, 0x04, 0x00, 0x14, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00,
  0x80, 0x00, 0x00, 0x00, 0xd8, 0x00, 0x00, 0x00, 0xc8, 0x43, 0x00, 0x00,
  0xd8, 0x43, 0x00, 0x00, 0x64, 0x4d, 0x00, 0x00, 0x03, 0x00, 0x00, 0x00,
  0x01, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00, 0x4e, 0xbb, 0xff, 0xff,
  0x0c, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00, 0x38, 0x00, 0x00, 0x00,
  0x0f, 0x00, 0x00, 0x00, 0x73, 0x65, 0x72, 0x76, 0x69, 0x6e, 0x67, 0x5f,
  0x64, 0x65, 0x66, 0x61, 0x75, 0x6c, 0x74, 0x00, 0x01, 0x00, 0x00, 0x00,
  0x04, 0x00, 0x00, 0x00, 0x9c, 0xff, 0xff, 0xff, 0x11, 0x00, 0x00, 0x00,
  0x04, 0x00, 0x00, 0x00, 0x06, 0x00, 0x00, 0x00, 0x6f, 0x75, 0x74, 0x70,
  0x75, 0x74, 0x00, 0x00, 0x01, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00,
  0x0e, 0xbc, 0xff, 0xff, 0x04, 0x00, 0x00, 0x00, 0x05, 0x00, 0x00, 0x00,
  0x69, 0x

In [25]:
!cat {MICRO_DIR}/micro_structured_pruned_threshold_0_6.cc

unsigned char g_model[] = {
  0x1c, 0x00, 0x00, 0x00, 0x54, 0x46, 0x4c, 0x33, 0x14, 0x00, 0x20, 0x00,
  0x1c, 0x00, 0x18, 0x00, 0x14, 0x00, 0x10, 0x00, 0x0c, 0x00, 0x00, 0x00,
  0x08, 0x00, 0x04, 0x00, 0x14, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00,
  0x80, 0x00, 0x00, 0x00, 0xd8, 0x00, 0x00, 0x00, 0xc8, 0x43, 0x00, 0x00,
  0xd8, 0x43, 0x00, 0x00, 0x64, 0x4d, 0x00, 0x00, 0x03, 0x00, 0x00, 0x00,
  0x01, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00, 0x4e, 0xbb, 0xff, 0xff,
  0x0c, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00, 0x38, 0x00, 0x00, 0x00,
  0x0f, 0x00, 0x00, 0x00, 0x73, 0x65, 0x72, 0x76, 0x69, 0x6e, 0x67, 0x5f,
  0x64, 0x65, 0x66, 0x61, 0x75, 0x6c, 0x74, 0x00, 0x01, 0x00, 0x00, 0x00,
  0x04, 0x00, 0x00, 0x00, 0x9c, 0xff, 0xff, 0xff, 0x11, 0x00, 0x00, 0x00,
  0x04, 0x00, 0x00, 0x00, 0x06, 0x00, 0x00, 0x00, 0x6f, 0x75, 0x74, 0x70,
  0x75, 0x74, 0x00, 0x00, 0x01, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00,
  0x0e, 0xbc, 0xff, 0xff, 0x04, 0x00, 0x00, 0x00, 0x05, 0x00, 0x00, 0x00,
  0x69, 0x

In [31]:
!cat {MICRO_DIR}/micro_structured_pruned_threshold_0_75.cc

unsigned char g_model[] = {
  0x1c, 0x00, 0x00, 0x00, 0x54, 0x46, 0x4c, 0x33, 0x14, 0x00, 0x20, 0x00,
  0x1c, 0x00, 0x18, 0x00, 0x14, 0x00, 0x10, 0x00, 0x0c, 0x00, 0x00, 0x00,
  0x08, 0x00, 0x04, 0x00, 0x14, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00,
  0x80, 0x00, 0x00, 0x00, 0xd8, 0x00, 0x00, 0x00, 0xc8, 0x43, 0x00, 0x00,
  0xd8, 0x43, 0x00, 0x00, 0x64, 0x4d, 0x00, 0x00, 0x03, 0x00, 0x00, 0x00,
  0x01, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00, 0x4e, 0xbb, 0xff, 0xff,
  0x0c, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00, 0x38, 0x00, 0x00, 0x00,
  0x0f, 0x00, 0x00, 0x00, 0x73, 0x65, 0x72, 0x76, 0x69, 0x6e, 0x67, 0x5f,
  0x64, 0x65, 0x66, 0x61, 0x75, 0x6c, 0x74, 0x00, 0x01, 0x00, 0x00, 0x00,
  0x04, 0x00, 0x00, 0x00, 0x9c, 0xff, 0xff, 0xff, 0x11, 0x00, 0x00, 0x00,
  0x04, 0x00, 0x00, 0x00, 0x06, 0x00, 0x00, 0x00, 0x6f, 0x75, 0x74, 0x70,
  0x75, 0x74, 0x00, 0x00, 0x01, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00,
  0x0e, 0xbc, 0xff, 0xff, 0x04, 0x00, 0x00, 0x00, 0x05, 0x00, 0x00, 0x00,
  0x69, 0x

In [37]:
!cat {MICRO_DIR}/micro_structured_pruned_threshold_0_85.cc

unsigned char g_model[] = {
  0x1c, 0x00, 0x00, 0x00, 0x54, 0x46, 0x4c, 0x33, 0x14, 0x00, 0x20, 0x00,
  0x1c, 0x00, 0x18, 0x00, 0x14, 0x00, 0x10, 0x00, 0x0c, 0x00, 0x00, 0x00,
  0x08, 0x00, 0x04, 0x00, 0x14, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00,
  0x80, 0x00, 0x00, 0x00, 0xd8, 0x00, 0x00, 0x00, 0xc8, 0x43, 0x00, 0x00,
  0xd8, 0x43, 0x00, 0x00, 0x64, 0x4d, 0x00, 0x00, 0x03, 0x00, 0x00, 0x00,
  0x01, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00, 0x4e, 0xbb, 0xff, 0xff,
  0x0c, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00, 0x38, 0x00, 0x00, 0x00,
  0x0f, 0x00, 0x00, 0x00, 0x73, 0x65, 0x72, 0x76, 0x69, 0x6e, 0x67, 0x5f,
  0x64, 0x65, 0x66, 0x61, 0x75, 0x6c, 0x74, 0x00, 0x01, 0x00, 0x00, 0x00,
  0x04, 0x00, 0x00, 0x00, 0x9c, 0xff, 0xff, 0xff, 0x11, 0x00, 0x00, 0x00,
  0x04, 0x00, 0x00, 0x00, 0x06, 0x00, 0x00, 0x00, 0x6f, 0x75, 0x74, 0x70,
  0x75, 0x74, 0x00, 0x00, 0x01, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00,
  0x0e, 0xbc, 0xff, 0xff, 0x04, 0x00, 0x00, 0x00, 0x05, 0x00, 0x00, 0x00,
  0x69, 0x

In [43]:
!cat {MICRO_DIR}/micro_structured_pruned_threshold_0_95.cc

unsigned char g_model[] = {
  0x1c, 0x00, 0x00, 0x00, 0x54, 0x46, 0x4c, 0x33, 0x14, 0x00, 0x20, 0x00,
  0x1c, 0x00, 0x18, 0x00, 0x14, 0x00, 0x10, 0x00, 0x0c, 0x00, 0x00, 0x00,
  0x08, 0x00, 0x04, 0x00, 0x14, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00,
  0x80, 0x00, 0x00, 0x00, 0xd8, 0x00, 0x00, 0x00, 0xc8, 0x43, 0x00, 0x00,
  0xd8, 0x43, 0x00, 0x00, 0x64, 0x4d, 0x00, 0x00, 0x03, 0x00, 0x00, 0x00,
  0x01, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00, 0x4e, 0xbb, 0xff, 0xff,
  0x0c, 0x00, 0x00, 0x00, 0x1c, 0x00, 0x00, 0x00, 0x38, 0x00, 0x00, 0x00,
  0x0f, 0x00, 0x00, 0x00, 0x73, 0x65, 0x72, 0x76, 0x69, 0x6e, 0x67, 0x5f,
  0x64, 0x65, 0x66, 0x61, 0x75, 0x6c, 0x74, 0x00, 0x01, 0x00, 0x00, 0x00,
  0x04, 0x00, 0x00, 0x00, 0x9c, 0xff, 0xff, 0xff, 0x11, 0x00, 0x00, 0x00,
  0x04, 0x00, 0x00, 0x00, 0x06, 0x00, 0x00, 0x00, 0x6f, 0x75, 0x74, 0x70,
  0x75, 0x74, 0x00, 0x00, 0x01, 0x00, 0x00, 0x00, 0x04, 0x00, 0x00, 0x00,
  0x0e, 0xbc, 0xff, 0xff, 0x04, 0x00, 0x00, 0x00, 0x05, 0x00, 0x00, 0x00,
  0x69, 0x